In [ ]:
import os
import shutil
import tempfile
import zipfile
import pathlib as Path
from concurrent.futures import ThreadPoolExecutor

import scanpy as sc
import hisepy as hp
from spatialdata_io import xenium

import lamindb as ln

ln.track("HwRrBzGZpyOU")

In [ ]:
OUTS_DIR = '/home/workspace/PCD_data/xenium/commercial/data/outs'

os.makedirs(OUTS_DIR, exist_ok=True)

In [ ]:
SAMPLES = {
    'TIS05393':     '70f7c0d6-8d7e-4b86-acf3-ca80137fd849',  # Multiple Myeloma
    'TIS04788':     '38fe47a0-c09f-4373-9a8a-8a957c40aa9f',  # Multiple Myeloma
    'TIS05392':     '0c879d26-970f-47fc-a705-88b103b4de80',  # Multiple Myeloma
    'TIS05394':     'd7b4b14e-9856-4169-a363-213b4e29a99e',  # Multiple Myeloma
    'TIS04789':     '7c3113c3-23bf-4840-8f45-61505048322c',  # Multiple Myeloma
    'TIS05397_008': '16752455-bd23-4f88-8d05-2108e7778496',  # Normal BM (2024)
}

In [ ]:
ingest_label   = ln.ULabel(name='ingest').save()
raw_label      = ln.ULabel(name='raw').save()
analysis_label = ln.ULabel(name='commercial').save()

In [ ]:
def fetch_sample(name):
    sample_dir = os.path.join(OUTS_DIR, name)
    if os.path.isdir(sample_dir):
        return
    (zip_path, ) = hp.cache_files(file_ids=[SAMPLES[name]])
    with tempfile.TemporaryDirectory() as tmp:
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(tmp)
        extracted_folder = next(
            os.path.join(tmp, d) for d in os.listdir(tmp)
            if os.path.isdir(os.path.join(tmp, d))
        )
        shutil.move(extracted_folder, sample_dir)

with ThreadPoolExecutor(max_workers=16) as ex:
    list(ex.map(fetch_sample, SAMPLES))

In [ ]:
adatas = {}
for name in SAMPLES:
    sample_dir = os.path.join(OUTS_DIR, name)
    adata = xenium(
        sample_dir,
        morphology_focus=False,
        morphology_mip=False,
        cells_boundaries=False,
        nucleus_boundaries=False,
        cells_labels=False,
        nucleus_labels=False,
        transcripts=False,
    ).tables['table']
    adata.obs_names_make_unique()
    adata['tissue_id'] = name
    
    artifact = ln.Artifact.from_anndata(adata, key=f'{name}.h5ad', description='raw unprocessed h5ads from commercial outs').save()
    artifact.ulabels.add(ingest_label, raw_label, analysis_label)
    adatas[name] = adata

In [ ]:
combined = sc.concat(adatas, label='sample', join='inner')
combined.obs_names_make_unique()

artifact = ln.Artifact.from_anndata(
    combined,
    key='commercial_combined_raw.h5ad',
    description='all raw unprocessed h5ads from commercial outs combined. no further processing'
).save()
artifact.ulabels.add(ingest_label, raw_label, analysis_label)

ln.finish()